# 29 · 外部验证：换一批别人整理的病人

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → 12 第三个工具 ✅ → 13 标准接口 MCP ✅ → 14 固定流程 ✅ → 15 第一个 agent ✅ → 16 看错题 ✅ → 17 找相似病人 ✅ → 18 疾病介绍（RAG） ✅ → 19 基因工具 ✅ → 20 大考 ✅ → 21 正式评测 ✅ → 22 做成服务 ✅ → 23 和别人比 ✅ → 24 和大模型比 ✅ → 25 和 DeepRare 比 ✅ → 26 读病历 ✅ → 27 交付 ✅ → 28 时间外验证 ✅ → [29 外部验证 ✅]
```

> **这一章的代码在分支 `feat/temporal-external` 里**（和第 28 章同一个合并请求）。两组外部病人共 712 道题；DeepSeek 约 ¥30，OpenRouter 约 $6.5。**运行这本笔记不花钱**：考试的命令只列出来不重跑，结果从 MLflow 和逐题表里读。
>
> **这一章也分两段写。** 第 1–4 步是考之前写的，第 5 步起是考完以后填的。

## 背景：题没重复，但都是同一位老师出的

到第 28 章为止，所有考题都来自同一个地方：**Phenopacket Store**。留出集和开发集按论文分开了，题没有重复，但是：

- **同一批人、按同一套习惯整理**：哪种表现记成哪个 HPO 编号、记多细，都是一个团队的风格。"找相似病人"的病例库也出自这个团队，所以它在我们的题上可能比在真实世界里更准（[`docs/leakage.md`](../leakage.md) 第六节第 5 条早就写了这个担心）
- **都是发表过的病例报告**：论文里写的多是"这个病的典型病人"，真实门诊的病人常常不典型

**打个比方**：你一直做同一位老师出的题，题目没重复，但出题风格一样，你可能只是摸透了这位老师。换一位老师出题，才知道是不是真会。

**公司里的做法**：这叫**外部验证**（external validation）：换一个来源、换一批人整理的数据再考一次。和它相对，前面做的都是**内部验证**。医疗 AI 产品申请上市时，审评几乎一定会问"有没有在别的医院的数据上验证过"；临床预测模型的报告规范 TRIPOD 也把"内部验证"和"外部验证"分成不同的证据等级。

## 这一章做的 7 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 找外部数据 | RareBench，固定在 HuggingFace 的一次提交 | 别人整理的病人 |
| 2 | 查重叠：哪几组真算"外部" | 逐个比对症状 | 防止"外部"数据其实是同一批病人 |
| 3 | 读进来：诊断编号怎么认 | 多个编号、Orphanet 精确对应、没有论文的病人怎么考 | 判分规则事先说清 |
| 4 | 选题、先写方案 | 预注册：`docs/external_plan.md`；先算"地板" | 不看了结果再挑；分数和什么比 |
| 5 | 开考 | 每组 9–10 次考试，`--final`，泄露审计 | 只考一次 |
| 6 | 结果 | 置信区间、逐题配对、两种判法 | 回答方案里的问题 |
| 7 | 和内部验证比，掉了多少 | 同一个系统，两份题的分数并排 | 看出谁依赖 Phenopacket Store 的写法 |

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
for d in ("/opt/homebrew/bin", "/usr/local/bin", str(Path.home() / ".local/bin")):  # 编辑器不是从终端打开时，也找得到 uv
    if Path(d).is_dir() and d not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] += os.pathsep + d
import logging, warnings
warnings.filterwarnings("ignore", message="IProgress not found")   # 一条和我们无关的提示
logging.getLogger("mlflow").setLevel(logging.WARNING)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 找外部数据：RareBench

### 在干什么
找一批**不是 Phenopacket Store、也不是 HPO 整理资料时引用的论文**里来的病人。

### 背景
罕见病诊断领域，大家用来互相比较的公开数据不多。最常用的是 **RareBench**（清华大学等，KDD 2024），DeepRare 的论文也用它。它公开了四组病人：

| 组 | 人数 | 来源 |
|---|---|---|
| HMS | 88 | 德国汉诺威医学院一个门诊的病人（2019 年一项研究附带公开） |
| RAMEDIS | 624 | 医生提交病例的代谢病数据库（2010 年发表） |
| MME | 40 | 已发表的病例报告（Matchmaker Exchange 的测试集） |
| LIRICAL | 370 | 已发表的病例报告（LIRICAL 论文自己的测试集） |

第五组是北京协和医院（PUMCH）的病人：说明里写公开了其中 75 例，但 HuggingFace 上的数据包里没有这一组，所以用不了。

每个病人只有两样东西：**观察到的 HPO 编号**和**诊断编号**。没有性别，没有基因，没有出处论文。

### 用了什么技术，为什么
- **固定在 HuggingFace 的一次提交**（`6f054e0`），登记在 `evals/datasets.toml`，带校验码。和其他数据一个规矩
- 数据包本身就是几个 JSON 行文件，直接读，不需要它附带的加载脚本（新版 `datasets` 已经不再支持这种脚本）

### 公司里什么情况怎么做
- **用公开评测集之前，先读它的说明和原论文**：数据从哪来、怎么整理的、有没有和自己的训练数据重叠。别人用了，不等于它适合你的问题
- **外部数据和内部数据一样要固定版本**：HuggingFace 上的数据集也会更新

### 执行

In [ ]:
run("uv run raredx data fetch rarebench")
import json, zipfile
archive = zipfile.ZipFile(ROOT / "data/raw/rarebench/6f054e0/data.zip")
for name in ("HMS", "RAMEDIS", "MME", "LIRICAL"):
    lines = archive.read(f"data/{name}.jsonl").decode().splitlines()
    print(f"{name:<8} {len(lines):>4} 个病人")
print("\nHMS 第 9 个病人，原样：")
print(archive.read("data/HMS.jsonl").decode().splitlines()[8])

### 结果怎么看
- 四组一共 1,122 个病人
- HMS 第 9 个病人的原样：一串 HPO 编号，诊断写了两个 OMIM 编号（`OMIM:106300`、`OMIM:183840`），`Department` 是空的。**一个病人列了两个诊断编号**，这一点第 3 步要处理

---

## 第 2 步 · 查重叠：哪几组真算"外部"

### 在干什么
把 RareBench 每一个病人，和 Phenopacket Store 0.1.27 里**同一个病**的病人逐个比症状，看是不是同一个人。

### 背景
"外部数据"最怕的是：看着是另一份数据，其实是同一批病人换了个包装。LIRICAL 组和 MME 组都是从**已发表的病例报告**里整理的，Phenopacket Store 也是；同一篇病例报告，完全可能被两边都收进去。要是这样，用它考就不是"外部验证"，而且：

- 同一个病人如果在我们的**开发集**里，他就在"找相似病人"的病例库里，等于把答案放在了工具里
- RareBench 不写出处论文，我们**没法按论文遮挡**

### 用了什么技术，为什么
- 比法：两个病人的症状集合的重合程度（**Jaccard 相似度**：共有的症状数 ÷ 两人症状合起来的总数）。1 表示一模一样，0.8 以上基本可以认为是同一个人
- 先把编号换成同一版 HPO 的写法再比，免得同一个症状因为写法不同被当成两个

### 公司里什么情况怎么做
- **合并或引入外部数据集之前，先做去重**（deduplication）。机器学习里"训练集和测试集里有同一条数据"是最常见的评测事故之一，大模型评测里专门叫它"数据污染"（contamination）
- **完全一样**好查；**几乎一样**（同一个病人被两个团队各整理了一遍，编号略有出入）才是难点，所以要用相似度，不能只比是否相等

### 执行
下面在本机算，不花钱，约半分钟：

In [ ]:
from collections import Counter, defaultdict
from raredx.data import cases as store
from raredx.knowledge import hpo
knowledge = hpo.load(ROOT / "data/raw/hpo/v2026-09-01")
release = store.read(ROOT / "data/processed/phenopacket-store/0.1.27")
rarebench = store.read(ROOT / "data/processed/rarebench/6f054e0")
same_disease = defaultdict(list)
for c in release:
    same_disease[c.disease_id].append(c)

def closest(patient):
    """同一个病（任何一个算对的 OMIM 编号）的已发表病人里，症状最像的那一个，和它的相似度"""
    mine = set(patient.hpo_ids)
    best, who = 0.0, None
    for disease in (patient.disease_id, *patient.also_accepted):
        for other in same_disease.get(disease, []):
            theirs = {knowledge.current(t) for t in other.hpo_ids} - {None}
            j = len(mine & theirs) / len(mine | theirs)
            if j > best:
                best, who = j, other
    return best, who

print(f"{'组':<8}{'人数':>5}{'没有同病病人':>10}{'相似度<0.5':>10}{'0.5–0.8':>9}{'≥0.8':>6}{'其中在开发集':>9}")
for name in ("HMS", "RAMEDIS", "MME", "LIRICAL"):
    group = [c for c in rarebench if c.cohort == name]
    found = [closest(c) for c in group]
    none = sum(who is None for _, who in found)
    low = sum(who is not None and j < 0.5 for j, who in found)
    mid = sum(0.5 <= j < 0.8 for j, _ in found)
    high = [who for j, who in found if j >= 0.8]
    print(f"{name:<8}{len(group):>5}{none:>12}{low:>12}{mid:>9}{len(high):>6}{sum(w.split == 'design' for w in high):>11}")

lirical_270 = next(c for c in rarebench if c.id == "RareBench_LIRICAL_270")
j, who = closest(lirical_270)
print(f"\n例子：RareBench_LIRICAL_270 和 {who.id}（{who.split}）相似度 {j:.2f}")
print("  诊断：", lirical_270.disease_label, "|", who.disease_label)
print("  症状：", "、".join(lirical_270.hpo_labels))

### 结果怎么看

| 组 | 人数 | 和 Phenopacket Store 的病人几乎一样（≥0.8） | 结论 |
|---|---|---|---|
| HMS | 88 | 0（只有 5 人找得到同病病人，相似度都不到 0.5） | **外部** |
| RAMEDIS | 624 | 0（95 人找得到同病病人，相似度都不到 0.5） | **外部** |
| MME | 40 | 0（它的 17 种病，0.1.27 里一个病人都没有） | 不用：出自病例报告，HPO 可能引用了这些报告，又没有出处没法遮 |
| LIRICAL | 370 | **22**（12 个在开发集，10 个在留出集），另有 105 个在 0.5–0.8 | **不用：和我们是同一批来源** |

**例子**：RareBench_LIRICAL_270 和留出集里的 PMID_28371217_proband，**同一个病、同样 9 个症状，相似度 1.00**，就是同一个人。它要是被当成外部题，第 21 章的留出集里已经考过了；更糟的是另外 12 个，它们就在"找相似病人"的病例库里。

所以外部验证只用 **HMS 和 RAMEDIS**。

> **这一步改变了原来的计划。** 项目第一天的总体计划（`docs/00_plan.md`）写的是"RareBench 公开子集：MME、HMS、LIRICAL、RAMEDIS"四组都用。查了重叠才知道 LIRICAL 组不能算外部。"别人都这么用"不等于"适合我们"。

### 面试怎么说
> "Before using RareBench as an external set I matched every patient against our own store by phenotype overlap: its LIRICAL subset holds 22 patients that are near-identical to ours — 12 of them sitting in the case bank our similar-patients tool searches — so only the clinic and registry subsets, HMS and RAMEDIS, count as external." 

---

## 第 3 步 · 读进来：诊断编号怎么认

### 在干什么
把 RareBench 的病人读成项目统一的"病例"格式，重点是**诊断编号怎么认**。

### 背景
我们所有系统都用 OMIM 编号回答，判分也只认 OMIM 编号（第 07 章）。RareBench 的诊断写成三种编号：OMIM、Orphanet（ORPHA，欧洲的罕见病目录）、中国罕见病目录（CCRD），而且常常一个病人列好几个：

| 病人 | RareBench 写的 | 是什么情况 |
|---|---|---|
| HMS 第 9 个 | `OMIM:106300`、`OMIM:183840` | 两个 OMIM 编号 |
| RAMEDIS 第 1 个 | `OMIM:251000`、`ORPHA:27`、`CCRD:71` | 同一个病的三种写法 |
| HMS 第 24 个 | `ORPHA:81`（抗合成酶综合征） | **一个 OMIM 编号都没有**：它不是单基因遗传病 |

### 用了什么技术，为什么
规则写在 `src/raredx/data/rarebench.py`，有测试，**在任何系统考这份题之前定下**：

1. 病人列出的**每一个 OMIM 编号都算对**（答中任意一个就行）
2. 列出的 Orphanet 编号，如果 Orphanet 自己把它**精确对应**（"E"类）到某个 OMIM 编号，那个也算对；只是"上下级"的对应不算（一个大类不等于其中某一个病）
3. 一个 OMIM 编号都没有的病人，诊断就记成它自己的编号（比如 `ORPHA:81`）。用 OMIM 回答的系统**不可能答对**，算错。真实门诊里也是这样：我们的系统只认识单基因病
4. 症状编号换成我们这一版 HPO 的写法（别名换成正名，作废的换成替代它的）

另外两件事：
- **没有出处论文，就没有东西要遮**。这些病人不来自论文，HPO 的资料不可能引用他们。考试照样做泄露审计（把工具的每一个回答都读一遍），只是没有"病人自己的论文"可找
- 判分程序改成"答中任意一个算对的编号都算对"（以前一个病人只有一个编号）

> **一个真实踩过的坑**：第一版的"诊断名称"只取第一个认得的 OMIM 编号的名字。HMS 第 6 个病人列了 8 个编号，前面是"非乳头状肾细胞癌"，后面是好几个系统性红斑狼疮。按编号判分没问题（任意一个都算对），但 DeepRare 的**宽松判分**是把诊断名称交给大模型裁判看的，裁判会以为确诊是"肾细胞癌"。改成把所有编号的病名都写上，用分号连起来。DeepRare 自己读 RareBench 也是这么做的。
>
> **第二个坑**：这些病人的"出处论文"一栏是空的，考卷文件每行末尾就多了一个制表符。提交代码时，自动检查里有一条"删掉行尾空白"，把它当成多余的空格删了，考卷就读不出来了。改法：让这条检查跳过 `.tsv` 数据文件（表格最后一列是空的，那是一个值，不是空白）。

### 公司里什么情况怎么做
- **两个系统的"标签体系"不一样时，对应规则要事先写死、写进代码、有测试**，而且宁严勿宽。不然考完一看分数低，很容易"再放宽一点"
- **答不了的题照样算在分母里**，单独报告有多少。去掉它们会让分数好看，但那不是真实情况

### 执行

In [ ]:
run("uv run raredx data build rarebench")
rarebench = store.read(ROOT / "data/processed/rarebench/6f054e0")
by_id = {c.id: c for c in rarebench}
for case_id in ("RareBench_HMS_009", "RareBench_RAMEDIS_001", "RareBench_HMS_024", "RareBench_HMS_006"):
    c = by_id[case_id]
    print(f"{case_id}: 判分认 {[c.disease_id, *c.also_accepted]}")
    print(f"    名称：{c.disease_label[:150]}")

### 结果怎么看
- HMS 第 9 个：两个 OMIM 编号都算对
- RAMEDIS 第 1 个：只有一个 OMIM 编号；`ORPHA:27` 精确对应的也是它，不重复
- HMS 第 24 个：判分认的是 `ORPHA:81`，没有任何系统能用 OMIM 编号答中，注定答不对
- HMS 第 6 个：列了 8 个编号，其中 7 个 OMIM 编号都算对；第 8 个 `ORPHA:93552`（儿童系统性红斑狼疮）在 Orphanet 里没有精确对应的 OMIM 编号，不添。名称把 8 个都写上了

---

## 第 4 步 · 选题、先写方案

### 在干什么
定下三份考卷，写好方案提交：[`docs/external_plan.md`](../external_plan.md)。

### 考卷

| 考卷 | 题数 | 给谁用 |
|---|---|---|
| `rarebench-hms` | 88 | 所有系统 |
| `rarebench-ramedis` | 624 | 不花钱的系统和用 DeepSeek 的系统 |
| `rarebench-ramedis-200` | 200（按固定种子从上一份里抽） | 按题收费的系统：三家通用大模型、DeepRare。和第 24 章的 `heldout-papers-200` 同一个道理：已经考过全部 624 道的系统，直接取这 200 道的答案，不重考 |

### 先算"地板"
看分数之前，先问一句：**什么都不会的系统能拿多少？**

RAMEDIS 的 624 个病人里，**苯丙酮尿症有 148 个**。一个永远只回答"苯丙酮尿症"的系统，第 1 名就有 **23.7%**。所以 RAMEDIS 上的分数要和这个地板比，不能单看。

### 主要结论（只有一个）

> **在两组外部病人上（HMS + RAMEDIS，共 712 道），最终 agent 的第 1 名命中率，是否高于固定流程。**

这是第 21 章的主要结论，换到别人整理的病人上再验一次。它问的是整个项目最核心的问题：**让大模型自己决定查什么，比把工具按固定顺序跑一遍再交给它，是不是真的更好**。两组合起来算，是因为 HMS 只有 88 道，单独测不出第 21 章那种大小的差别。

### 我的预期（写在考之前）
- **HMS 整体分数会低**：四分之一注定答不对，剩下的多是风湿免疫病，HPO 对它们的记录不如单基因病详细
- **HMS 上通用大模型可能追上甚至超过 agent**：这些病是教科书上的常见罕见病，大模型很熟
- **RAMEDIS 分数会高**：代谢病的化验异常很有特点，agent 第 1 名 45%–65%
- **主要结论成立，但领先幅度比第 21 章小**

### 一件要事先说的事：钱不够
三家通用大模型考 HMS 和 RAMEDIS-200 大约要 $6.5，OpenRouter 账户里只有 $5.10。方案里写好了顺序：先考 HMS，再考 RAMEDIS-200，钱不够就停在那里，没考的照实写"没考"，**不换成更便宜的模型**（换了就不是同一个比较了）。

### 公司里什么情况怎么做
- **报告任何分数之前，先报"地板"**：最常见类别的比例、随机猜的分数。地板很高的数据集上，一个看着不错的分数可能什么都不说明
- **预算不够时，事先定好先做什么、后做什么**，而不是做到一半临时挑

### 执行

In [ ]:
from raredx.evals import splits
from raredx.evals.hits import answerable
for name in ("rarebench-hms", "rarebench-ramedis", "rarebench-ramedis-200"):
    exam = [by_id[case_id] for case_id, _ in splits.read(ROOT / f"evals/splits/{name}.tsv")]
    top, count = Counter(c.disease_label for c in exam).most_common(1)[0]
    lost = sum(not answerable(c, knowledge) for c in exam)
    print(f"{name:<22} {len(exam):>4} 道  {len({c.disease_id for c in exam}):>3} 种病  注定答不对 {lost:>2} 道   "
          f"最常见：{top[:28]} {count} 道（地板 {count / len(exam):.1%}）")

### 结果怎么看
- HMS：88 道、36 种病，**24 道注定答不对**；最常见的病（肉芽肿性多血管炎）占 12.5%
- RAMEDIS：624 道、74 种病，没有注定答不对的；**苯丙酮尿症占 23.7%**
- RAMEDIS-200：200 道、40 种病，苯丙酮尿症占 23.5%，和全部 624 道差不多

### 面试怎么说
> "For the external test I fixed the reading rules before any run — every listed OMIM entry counts, an Orphanet code only through Orphanet's exact mapping, a patient with no OMIM entry stays in the denominator as a miss — and I reported the floor first: a quarter of RAMEDIS is phenylketonuria, so always answering PKU scores 23.7%." 

---

**第 5 步起（开考、结果）考完以后补上。**